# General setup

In [1]:
# Imports
import matplotlib
matplotlib.use('Agg')
import argparse
import re
import json
import warnings
import numpy as np
from modules.CHILI import CHILI
from modules.net import SCVAE
from torch_geometric.loader import DataLoader
import torch
from torch.utils.data import TensorDataset
import datetime
import pathlib
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from ase import Atoms
from ase.io import write
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from modules.loss_functions import weighted_MSELoss, weighted_CrossEntropyLoss
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

In [2]:
# Functions
def create_cif(cell_params, cell_positions, cell_atoms, filename, prediction=True, composition=None, simplified_atom_identities=False):
    """
    Create a CIF file from the cell parameters, positions and atoms
    """
    if prediction:
        # Find argmax of atoms
        cell_atoms = np.argmax(cell_atoms, axis=1)

    # Remove atoms with atom number 0
    cell_positions = cell_positions[cell_atoms != 0]
    cell_atoms = cell_atoms[cell_atoms != 0]
    
    # Remove atoms not in the unit cell
    cell_atoms = cell_atoms[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    cell_positions = cell_positions[(cell_positions[:,0] < 0.95) & (cell_positions[:,1] < 0.95) & (cell_positions[:,2] < 0.95)]
    
    
    if simplified_atom_identities:
        cell_atoms = np.where(cell_atoms == 1, 8, cell_atoms)
        cell_atoms = np.where(cell_atoms == 2, 26, cell_atoms)
    
    # Create Atoms object
    atoms = Atoms(cell_atoms, scaled_positions=cell_positions, cell=cell_params)

    if not composition:
        composition = str(atoms.symbols)

    # Write CIF
    write(filename + f'.cif', images=atoms, format='cif') # _{composition}

    if not prediction:
        return composition
    return None

In [3]:
# Setup
model_path = './models/Interpolation_v2_Unitcell_latentLog_2d_latentMSE_biggerDecoder/' # './models/Supercell_latentLog_beta_annealing_3d_latentMSE_biggerDecoder_v2/'
model_name = 'best_model.pth' # 'best_model_annealed.pth'    'best_model.pth'
setup_json_path = f'{model_path}setup_json.json'
with open(setup_json_path, 'r') as setup_json_file:
    setup_json = json.load(setup_json_file)

# latent_space_version = '_prior' # '' or '_prior'

# Make prediction folders
experimental_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions'
pathlib.Path(experimental_folder).mkdir(parents=True, exist_ok=True)

interpolation_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions'
pathlib.Path(interpolation_folder).mkdir(parents=True, exist_ok=True)

sample_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions'
pathlib.Path(sample_folder).mkdir(parents=True, exist_ok=True)

# Make paper figures folder
paper_figures_folder = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures'
pathlib.Path(paper_figures_folder).mkdir(parents=True, exist_ok=True)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# Load model
model = SCVAE(
    latent_dim=setup_json['model']['latent_dim'],
    out_dim=setup_json['model']['out_dim'],
    prior_factor=setup_json['model']['prior_factor'],
    gnn_dim=setup_json['model']['gnn_dim'],
    gnn_heads=setup_json['model']['gnn_heads'],
    gnn_edge_dim=setup_json['model']['gnn_edge_dim'],
    scattering_channels=setup_json['model']['scattering_channels'],
    scattering_dim=setup_json['model']['scattering_dim'],
    scattering_kernel_size=setup_json['model']['scattering_kernel_size'],
    scattering_stride=setup_json['model']['scattering_stride'],
    scattering_padding=setup_json['model']['scattering_padding'],
    composition_dim=setup_json['model']['composition_dim'],
    decoder_hidden_dim=setup_json['model']['decoder_hidden_dim'],
    position_output_dim=setup_json['model']['position_output_dim'],
    atom_output_dim=setup_json['model']['atom_output_dim'],
    cell_output_dim=setup_json['model']['cell_output_dim'],
    seperate_decoder=setup_json['training']['seperate_decoder'],
).to(device)

# Load model weights
model.load_state_dict(torch.load(model_path + model_name))

<All keys matched successfully>

In [5]:
# Load normalization parameters
if setup_json['data']['normalize_cell_parameters']:
    cell_means = torch.tensor([
        setup_json['data']['cell_normalization']['a']['mean'],
        setup_json['data']['cell_normalization']['b']['mean'],
        setup_json['data']['cell_normalization']['c']['mean'],
        setup_json['data']['cell_normalization']['alpha']['mean'],
        setup_json['data']['cell_normalization']['beta']['mean'],
        setup_json['data']['cell_normalization']['gamma']['mean'],
    ]).float().to(device)
    cell_stds = torch.tensor([
        setup_json['data']['cell_normalization']['a']['std'],
        setup_json['data']['cell_normalization']['b']['std'],
        setup_json['data']['cell_normalization']['c']['std'],
        setup_json['data']['cell_normalization']['alpha']['std'],
        setup_json['data']['cell_normalization']['beta']['std'],
        setup_json['data']['cell_normalization']['gamma']['std'],
    ]).float().to(device)

if setup_json['data']['normalize_atom_positions']:
    atom_position_means = torch.tensor(setup_json['data']['atom_position_normalization']['mean']).float().to(device)
    atom_position_stds = torch.tensor(setup_json['data']['atom_position_normalization']['std']).float().to(device)

if setup_json['data']['normalize_distances']:
    distance_means = torch.tensor(setup_json['data']['distance_normalization']['mean']).float().to(device)
    distance_stds = torch.tensor(setup_json['data']['distance_normalization']['std']).float().to(device)

beta = setup_json['training']['beta']
out_dim = setup_json['model']['out_dim']

# Load model test results

## Code

In [6]:
# Load results from test script
with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/losses_{model_name[:-4]}.json', 'r') as f:
    losses_json = json.load(f)
df_losses = pd.DataFrame(losses_json)

with open(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/CrystalTypeAnalysis/reconstructions_{model_name[:-4]}.json', 'r') as f:
    rec_json = json.load(f)
df_rec = pd.DataFrame(rec_json)
# Capitalize crystal types if not already
name_map = {'interpolated': 'Interpolated', 'rocksalt': 'RockSalt', 'spinel': 'Spinel', 'zincblende': 'ZincBlende'}
df_rec['crystalType'] = df_rec['crystalType'].apply(lambda x: name_map[x] if x in name_map else x)

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    df_rec[['ls1', 'ls2']] = df_rec['latent_space_mean'].apply(pd.Series)
    df_rec[['ls1_prior', 'ls2_prior']] = df_rec['latent_space_mean_prior'].apply(pd.Series)
else:
    # Perform PCA
    pca = PCA(n_components=2)
    pca.fit(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1', 'pc2']] = pca.transform(np.array(df_rec['latent_space_mean'].values.tolist()))
    df_rec[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_rec['latent_space_mean_prior'].values.tolist()))

In [7]:
df_losses.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,crystalType,particleSize
0,-2.951440,0.043889,0.003762,0.012949,0.027177,0.016461,interpolated,12.284917
1,-4.335351,0.006305,0.001460,0.004185,0.000660,0.013323,interpolated,44.640343
2,-3.642628,0.023067,0.010591,0.004659,0.007817,0.006134,interpolated,44.011925
3,-4.366133,0.005730,0.000481,0.004274,0.000975,0.013598,interpolated,53.778534
4,-4.128576,0.014552,0.000598,0.007321,0.006632,0.002985,interpolated,13.161923


In [8]:
df_rec.head()

,crystalType,n_atoms,n_oxygens,n_metals,cell_parameters,cell_positions,cell_atoms,latent_space_mean,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,Interpolated,30,16,14,"[6.332734107971191, 6.3343329429626465, 10.102...","[[0.16720999777317047, 0.3318899869918823, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-0.5473625063896179, -0.1996995508670807]","[0.045702069997787476, 0.04858894646167755]","[-0.5249666571617126, -0.20821890234947205]","[0.04666437581181526, 0.050894640386104584]","[-0.09482824802398682, -0.09482824802398682, 0...","[[0.16665999591350555, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-0.547363,-0.199700,-0.524967,-0.208219
1,Interpolated,25,16,9,"[7.492743968963623, 7.494154930114746, 11.3054...","[[0.1632000058889389, 0.32537001371383667, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.8440706729888916, 1.3497462272644043]","[0.04379792511463165, 0.052160877734422684]","[0.8288419842720032, 1.3665492534637451]","[0.0459008663892746, 0.05443200096487999]","[1.6037931442260742, 1.6037931442260742, 1.913...","[[0.16665999591350555, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",0.844071,1.349746,0.828842,1.366549
2,Interpolated,30,16,14,"[6.037561893463135, 6.038360595703125, 9.85054...","[[0.16639000177383423, 0.33344000577926636, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-0.9294614195823669, -0.4399583637714386]","[0.044246524572372437, 0.04572879523038864]","[-0.9198840856552124, -0.450309157371521]","[0.04439115896821022, 0.04558873176574707]","[-0.5019283294677734, -0.5019283294677734, 0.1...","[[0.16666999459266663, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-0.929461,-0.439958,-0.919884,-0.450309
3,Interpolated,27,16,11,"[5.790594100952148, 5.792788505554199, 9.20223...","[[0.16798999905586243, 0.3361400067806244, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-2.042084217071533, -0.7786155939102173]","[0.055540766566991806, 0.05727757140994072]","[-2.068140745162964, -0.7757939696311951]","[0.056215379387140274, 0.057240553200244904]","[-0.812208890914917, -0.812208890914917, -0.96...","[[0.16666999459266663, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-2.042084,-0.778616,-2.068141,-0.775794
4,Interpolated,29,16,13,"[5.6408257484436035, 5.639517784118652, 9.0872...","[[0.17156000435352325, 0.33845001459121704, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-2.224567413330078, -1.4447990655899048]","[0.06936421990394592, 0.07205921411514282]","[-2.2326500415802, -1.4581630229949951]","[0.07066599279642105, 0.07259181886911392]","[-1.0607212781906128, -1.0607212781906128, -1....","[[0.16666999459266663, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-2.224567,-1.444799,-2.232650,-1.458163


In [9]:
df_combined = pd.concat([df_losses.drop('crystalType', axis=1), df_rec], axis=1)
df_combined.head()

,total,reconstruction_loss,cell_parameters,cell_positions,cell_atoms,kld,particleSize,crystalType,n_atoms,n_oxygens,...,latent_space_std,latent_space_mean_prior,latent_space_std_prior,true_cell_parameters,true_cell_positions,true_cell_atoms,ls1,ls2,ls1_prior,ls2_prior
0,-2.951440,0.043889,0.003762,0.012949,0.027177,0.016461,12.284917,Interpolated,30,16,...,"[0.045702069997787476, 0.04858894646167755]","[-0.5249666571617126, -0.20821890234947205]","[0.04666437581181526, 0.050894640386104584]","[-0.09482824802398682, -0.09482824802398682, 0...","[[0.16665999591350555, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-0.547363,-0.199700,-0.524967,-0.208219
1,-4.335351,0.006305,0.001460,0.004185,0.000660,0.013323,44.640343,Interpolated,25,16,...,"[0.04379792511463165, 0.052160877734422684]","[0.8288419842720032, 1.3665492534637451]","[0.0459008663892746, 0.05443200096487999]","[1.6037931442260742, 1.6037931442260742, 1.913...","[[0.16665999591350555, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",0.844071,1.349746,0.828842,1.366549
2,-3.642628,0.023067,0.010591,0.004659,0.007817,0.006134,44.011925,Interpolated,30,16,...,"[0.044246524572372437, 0.04572879523038864]","[-0.9198840856552124, -0.450309157371521]","[0.04439115896821022, 0.04558873176574707]","[-0.5019283294677734, -0.5019283294677734, 0.1...","[[0.16666999459266663, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-0.929461,-0.439958,-0.919884,-0.450309
3,-4.366133,0.005730,0.000481,0.004274,0.000975,0.013598,53.778534,Interpolated,27,16,...,"[0.055540766566991806, 0.05727757140994072]","[-2.068140745162964, -0.7757939696311951]","[0.056215379387140274, 0.057240553200244904]","[-0.812208890914917, -0.812208890914917, -0.96...","[[0.16666999459266663, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-2.042084,-0.778616,-2.068141,-0.775794
4,-4.128576,0.014552,0.000598,0.007321,0.006632,0.002985,13.161923,Interpolated,29,16,...,"[0.06936421990394592, 0.07205921411514282]","[-2.2326500415802, -1.4581630229949951]","[0.07066599279642105, 0.07259181886911392]","[-1.0607212781906128, -1.0607212781906128, -1....","[[0.16666999459266663, 0.3333300054073334, 0.1...","[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, ...",-2.224567,-1.444799,-2.232650,-1.458163


In [10]:
# Load latent log if it exists
import yaml
try:
    df_latent_log = pd.read_csv(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_log.csv')

    df_latent_log = df_latent_log.drop(df_latent_log[df_latent_log['posterior_mean'] == 'posterior_mean'].index)

    # df_latent_log = df_latent_log[:1000]

    # Select only every tenth epoch
    # Convert epoch to ints
    df_latent_log['epoch'] = df_latent_log['epoch'].astype(np.int16)

    df_latent_log = df_latent_log[df_latent_log['epoch'] % 10 == 0]

    df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']] = df_latent_log[['posterior_mean', 'posterior_std', 'prior_mean', 'prior_std']].applymap(yaml.safe_load)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_latent_log[['ls1', 'ls2']] = df_latent_log['posterior_mean'].apply(pd.Series)
        df_latent_log[['ls1_prior', 'ls2_prior']] = df_latent_log['prior_mean'].apply(pd.Series)
    else:
        # Perform PCA
        # pca = PCA(n_components=2)
        # pca.fit(np.array(df_latent_log['posterior_mean'].values.tolist()))
        df_latent_log[['pc1', 'pc2']] = pca.transform(np.array(df_latent_log['posterior_mean'].values.tolist()))

        df_latent_log[['pc1_prior', 'pc2_prior']] = pca.transform(np.array(df_latent_log['prior_mean'].values.tolist()))
        
    print(df_latent_log.columns)
except:
    df_latent_log = None
    print('No latent log found')

Index(['epoch', 'posterior_mean', 'posterior_std', 'prior_mean', 'prior_std',
       'np_size', 'crystal_type', 'crystal_system', 'space_group', 'ls1',
       'ls2', 'ls1_prior', 'ls2_prior'],
      dtype='object')


## Plot

In [11]:
# Set seaborn style
# sns.set_theme(style='darkgrid', font_scale=1.)
# Make animation of latent space throughout 
# Plot latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_means.pdf', dpi=300)

In [12]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_combined[contour_column].min()
max_loss = df_combined[contour_column].max()

# Plot
if len(df_combined['latent_space_mean'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['ls1'].min()*1.1, df_combined['ls1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['ls2'].min()*1.1, df_combined['ls2'].max()*1.1, 1000)
    zi = griddata((df_combined['ls1'], df_combined['ls2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_combined['pc1'].min()*1.1, df_combined['pc1'].max()*1.1, 1000)
    yi = np.linspace(df_combined['pc2'].min()*1.1, df_combined['pc2'].max()*1.1, 1000)
    zi = griddata((df_combined['pc1'], df_combined['pc2']), df_combined[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 100, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_combined, x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()


In [13]:
# # 2D scatter plot with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         pass
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace)
        
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types):
#             fig.data[i]['visible'] = True
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j, crystal_type in enumerate(crystal_types):
#                 step['args'][1][i*n_crystal_types + j] = True
#             steps.append(step)    
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.show()

In [14]:
# Sample n points from each distribution in the latent space log
if df_latent_log is not None:
    n_samples = 100
    
    crystal_type_list = []
    latent_space_list = []
    epoch_list = []
    for latent_mean, latent_std, crystal_type, epoch in zip(df_latent_log['posterior_mean'], df_latent_log['posterior_std'], df_latent_log['crystal_type'], df_latent_log['epoch']):
        latent_space_samples = torch.distributions.Normal(loc=torch.tensor(latent_mean), scale=torch.tensor(latent_std)).sample((n_samples,)).numpy()
        latent_space_list.extend(latent_space_samples)
        crystal_type_list.extend([crystal_type] * n_samples)
        epoch_list.extend([epoch] * n_samples)

    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        df_log_samples = pd.DataFrame(latent_space_list, columns=['ls1', 'ls2'])
    else:
        latent_space_list = pca.transform(latent_space_list)
        df_log_samples = pd.DataFrame(latent_space_list, columns=['pc1', 'pc2'])

    df_log_samples['crystalType'] = crystal_type_list
    df_log_samples['epoch'] = epoch_list

In [15]:
# # 2D scatter plot and 2D histogram with slider

# symbols = ['circle', 'x', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagon', 'pentagon', 'hexagram', 'octagon', 'star-triangle-up', 'star-triangle-down', 'star-square', 'star-diamond', 'star-cross', 'hourglass', 'bowtie', 'circle-cross', 'circle-x', 'circle-star', 'circle-triangle-up', 'circle-triangle-down', 'circle-diamond', 'cross-thin', 'x-thin', 'square-thin', 'diamond-thin', 'cross-dot', 'x-dot', 'square-dot', 'diamond-dot', 'cross-open', 'x-open', 'square-open', 'diamond-open', 'cross-thin-open', 'x-thin-open', 'square-thin-open', 'diamond-thin-open', 'pentagon-open', 'hexagon-open', 'hexagram-open', 'octagon-open', 'star-open', 'star-triangle-up-open', 'star-triangle-down-open', 'star-square-open', 'star-diamond-open', 'star-cross-open', 'hourglass-open', 'bowtie-open', 'circle-cross-open', 'circle-x-open', 'circle-star-open', 'circle-triangle-up-open', 'circle-triangle-down-open', 'circle-diamond-open']
# color_scale = px.colors.qualitative.Dark24

# if df_latent_log is not None:
#     crystal_types = df_latent_log['crystal_type'].unique()
#     n_crystal_types = len(crystal_types)
#     trace_list = []
#     if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['ls1'],
#                     y=df_crystal['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['ls1'],
#                     y=df_crystal_samples['ls2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
            
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)
            
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='LS dim 1')
#         fig.update_yaxes(title_text='LS dim 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
#     else:
#         for epoch in df_latent_log['epoch'].unique():
#             trace_means = []
#             df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
#             df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
#             for i, crystal_type in enumerate(crystal_types):
#                 df_crystal = df_epoch[df_epoch['crystal_type'] == crystal_type]
#                 df_crystal_samples = df_epoch_samples[df_epoch_samples['crystalType'] == crystal_type]
#                 trace = go.Scatter(
#                     x=df_crystal['pc1'],
#                     y=df_crystal['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         size=10,
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         line=dict(
#                             color='black',
#                             width=1,
#                         ),
#                     ),
#                     visible=False,
#                 )
                
                
#                 # Sampled points
#                 trace_samples = go.Scatter(
#                     x=df_crystal_samples['pc1'],
#                     y=df_crystal_samples['pc2'],
#                     mode='markers',
#                     name=crystal_type,
#                     marker=dict(
#                         color=color_scale[i],
#                         symbol=symbols[i],
#                         opacity=0.5,
#                     ),
#                     visible=False,
#                 )
#                 trace_list.append(trace_samples)
#                 trace_means.append(trace)
#             trace_list.extend(trace_means)
            
#         fig = go.Figure(data=trace_list)
        
#         for i in range(n_crystal_types*2):
#             fig.data[i]['visible'] = True
        
#         n_steps = len(df_latent_log['epoch'].unique())
        
#         # Add slider
#         steps = []
#         for i, epoch in enumerate(df_latent_log['epoch'].unique()):
#             step = dict(
#                 method='restyle',
#                 args=['visible', [False] * len(trace_list)],
#                 label=f'{epoch}',
#             )
#             for j in range(n_crystal_types*2):
#                 step['args'][1][i*n_crystal_types*2 + j] = True
                
#             steps.append(step)  
        
#         sliders = [dict(
#             active=0,
#             currentvalue={"prefix": "Epoch: "},
#             pad={"t": 50},
#             steps=steps,
#         )]
        
#         fig.update_layout(
#             sliders=sliders,
#             width=1200,
#             height=800,
#             )
        
#         # Set axis labels
#         fig.update_xaxes(title_text='PC 1')
#         fig.update_yaxes(title_text='PC 2')
        
#         fig.write_html(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/latent_space_samples.html')
    

In [16]:
# Plot the latent space at specific epochs
epoch_to_plot = 0

if df_latent_log is not None:
    plt.figure(figsize=(10, 8))
    df_epoch = df_latent_log[df_latent_log['epoch'] == epoch_to_plot]
    df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch_to_plot]
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        sns.scatterplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['ls1'].min(), df_log_samples['ls1'].max())
        plt.ylim(df_log_samples['ls2'].min(), df_log_samples['ls2'].max())  
    else:
        sns.scatterplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20', alpha=0.5, legend=False)
        sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=100, palette='tab20', edgecolor='black')
        
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        
        plt.xlim(df_log_samples['pc1'].min(), df_log_samples['pc1'].max())
        plt.ylim(df_log_samples['pc2'].min(), df_log_samples['pc2'].max())
    
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    # plt.axis('equal')
    plt.tight_layout()
    plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epoch_{epoch_to_plot}.pdf', dpi=300)

In [17]:
# Make a combined figure of the latent space at different epochs
epochs_to_plot = [0, 10, 100, 350, 690, 890]
subplot_rows = 2
subplot_cols = 3
figsize = (10, 6.5)

if df_latent_log is not None:
    fig, ax = plt.subplots(subplot_rows, subplot_cols, figsize=figsize, sharex=True, sharey=True)
    ax = ax.flatten()
    
    if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
        xlim_min = df_log_samples['ls1'].min()
        xlim_max = df_log_samples['ls1'].max()
        ylim_min = df_log_samples['ls2'].min()
        ylim_max = df_log_samples['ls2'].max()
    else:
        xlim_min = df_log_samples['pc1'].min()
        xlim_max = df_log_samples['pc1'].max()
        ylim_min = df_log_samples['pc2'].min()
        ylim_max = df_log_samples['pc2'].max()
    
    for i, epoch in enumerate(epochs_to_plot):
        df_epoch = df_latent_log[df_latent_log['epoch'] == epoch]
        df_epoch_samples = df_log_samples[df_log_samples['epoch'] == epoch]
        
        if len(df_latent_log['posterior_mean'].iloc[0]) == 2:
            sns.histplot(data=df_epoch_samples, x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='ls1', y='ls2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        else:
            sns.histplot(data=df_epoch_samples, x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75, legend=False, ax=ax[i])
            sns.scatterplot(data=df_epoch, x='pc1', y='pc2', hue='crystal_type', style='crystal_type', s=50, palette='tab20', edgecolor='black', ax=ax[i])
            
            ax[i].set_xlabel('PC 1', fontsize=14, fontweight='bold')
            ax[i].set_ylabel('PC 2', fontsize=14, fontweight='bold')
            
            ax[i].set_xlim(xlim_min, xlim_max)
            ax[i].set_ylim(ylim_min, ylim_max)
        if i == 0:
            handles, labels = ax[i].get_legend_handles_labels()
            ax[i].legend(loc='lower center', bbox_to_anchor=(subplot_cols/2, 1), ncol=4)
        else:
            ax[i].legend([],[], frameon=False)
        ax[i].set_title(f' Epoch {epoch}', fontsize=12, fontweight='bold', y=1.0, pad=-15, loc='left')
    
    # Remove empty axes
    for i in range(len(epochs_to_plot), subplot_rows*subplot_cols):
        fig.delaxes(ax[i])
        
    # fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 3), ncols=5)
    
    plt.tight_layout()
    
    # Remove whitespace
    plt.subplots_adjust(wspace=0.02, hspace=0.02)
    
    plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_epochs.pdf', dpi=300)

# Experimental data

## Code

In [18]:
data_folder = './data/Experimental/'

# Load experimental data
data_paths = [str(p) for p in pathlib.Path(data_folder).rglob('*.gr')]

data_filepath = []
data_composition_string = []
data_composition = []
data_pdf = []
for data_path in data_paths:
    with open(data_path, 'r') as f:
        # Load data
        line_counter = 0
        for line in f:
            if line.startswith('composition'):
                composition = ''.join(line.split(' ')[2:])
            if line.startswith('0'):
                header_line = line_counter
                break
            line_counter += 1
    # Remove line breaks
    composition = composition.replace('\n', '')
    
    # Save composition string
    composition_string = deepcopy(composition)

    # Remove stochiometry from composition
    composition = re.sub(r'[0-9\.]+', '', composition)

    # # Split string on capital letters
    composition = re.findall('[A-Z][^A-Z]*', composition)

    # Translate composition to atom numbers
    composition = Atoms(symbols=composition).get_atomic_numbers()
    
    composition_onehot = np.zeros(119)
    composition_onehot[composition] = 1
    
    
    # Load data
    data = pd.read_csv(data_path, sep=' ', skiprows=header_line, names=['r [Å]', 'G(r) [Å⁻²]'])
    
    data_r = np.arange(0,60,0.01)
    data_Gr = np.interp(data_r, data['r [Å]'], data['G(r) [Å⁻²]'], left=0, right=0)
    data_Gr = data_Gr / np.amax(data_Gr)
    
    data_filepath.append(data_path)
    data_composition_string.append(composition_string)
    data_composition.append(composition_onehot)
    data_pdf.append(data_Gr)

# Convert to tensors
data_composition = torch.tensor(data_composition, dtype=torch.long)
data_pdf = torch.tensor(data_pdf, dtype=torch.float32)
data_composition_string_index = torch.tensor(np.arange(len(data_composition_string)))
data_filepath_index = torch.tensor(np.arange(len(data_filepath)))

exp_data = TensorDataset(data_pdf, data_composition, data_composition_string_index, data_filepath_index)

# Dataloader
exp_loader = DataLoader(exp_data, batch_size=10, shuffle=False)

# Results dict
exp_results = {
    'composition': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [19]:
# Inference
model.eval()
for batch in tqdm(exp_loader, desc='Inference', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    pdf, composition, composition_string_index, filepath_index = batch
    pdf = pdf.unsqueeze(-1).to(device)
    composition = composition.float().to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store composition
    for index in composition_string_index:
        exp_results['composition'].append(data_composition_string[index])
    
    # Store PDF
    exp_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    exp_results['prior_mean'].extend(prior_mean.cpu().tolist())
    exp_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    exp_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    exp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    exp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    exp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
                prediction=True,
                composition=data_composition_string[composition_string_index[batch_index]],
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_exp_results = pd.DataFrame(exp_results)
if len(df_exp_results['prior_mean'].iloc[0]) == 2:
    df_exp_results[['ls1', 'ls2']] = df_exp_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_exp_results[['pc1', 'pc2']] = pca.transform(np.array(df_exp_results['prior_mean'].values.tolist()))
df_exp_results.head()

# Drop IrO2
# df_exp_results = df_exp_results[df_exp_results['composition'] != 'IrO2']

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[-1.593813180923462, -0.6405230164527893]","[0.09613613039255142, 0.09387928247451782]","[-1.717536449432373, -0.6064298152923584]","[5.897527694702148, 5.899387836456299, 9.38331...","[[0.16680000722408295, 0.3352000117301941, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.593813,-0.640523
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-2.1372663974761963, -0.21823544800281525]","[0.11543595790863037, 0.11163424700498581]","[-2.1938061714172363, -0.07631607353687286]","[5.989924907684326, 5.992015838623047, 9.24749...","[[0.167480006814003, 0.33520999550819397, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-2.137266,-0.218235
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[-1.45340895652771, -0.6876246333122253]","[0.09248798340559006, 0.09030269831418991]","[-1.524922490119934, -0.6281325221061707]","[5.898794174194336, 5.898250579833984, 9.50557...","[[0.16633999347686768, 0.334089994430542, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.453409,-0.687625
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[0.19988952577114105, 0.21497651934623718]","[0.06774086505174637, 0.07994374632835388]","[0.14612111449241638, 0.13758671283721924]","[6.847782135009766, 6.845365047454834, 10.4790...","[[0.1694899946451187, 0.33445000648498535, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.199890,0.214977
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.004733748733997345, -0.8593271374702454]","[0.06259611994028091, 0.07178624719381332]","[0.03918557986617088, -0.8405420780181885]","[6.002366542816162, 6.010383129119873, 9.73057...","[[0.08987999707460403, 0.17062999308109283, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-0.004734,-0.859327


In [20]:
df_exp_results['ground_truth_crystal_type'] = None
df_exp_results['ground_truth_crystal_type'].iloc[0:3] = 'Rutile'
df_exp_results['ground_truth_crystal_type'].iloc[3:6] = 'Fluorite'
df_exp_results['ground_truth_crystal_type'].iloc[6:11] = 'Spinel'

## Plot

In [21]:
# Plot latent space placement of experimental data
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='ls1', y='ls2', hue='composition', style='ground_truth_crystal_type', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_exp_results, x='pc1', y='pc2', c='black', style='composition', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_samples.pdf', dpi=300)

In [22]:
# Plot the experimental data
# index_list = [0,1,2] # Rutile samples
index_list = [3, 4, 5] # Fluorite samples
# index_list = list(range(6,11)) # Spinel samples

plt.figure(figsize=(8, 6))
for index in index_list:
    plt.plot(np.arange(0,60,0.01), np.array(df_exp_results['pdf'].iloc[index]) + (index * 1.25), label=df_exp_results['composition'].iloc[index])
plt.xlabel('r [Å]', fontsize=14, fontweight='bold')
plt.ylabel('G(r) [A.U.]', fontsize=14, fontweight='bold')
plt.yticks([])
# plt.legend(ncols=2, loc='lower center', bbox_to_anchor=(0.5, 1.01))
plt.legend(ncols=3, loc='lower center', bbox_to_anchor=(0.5, 1.01))
# plt.title(f'{df_exp_results["composition"].iloc[index]}', fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Rutile.pdf', dpi=300)
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Fluorite.pdf', dpi=300)
# plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/experimental_data_Spinel.pdf', dpi=300)

# Interpolate in latent space

## Code

In [24]:
# Create latent samples for interpolation between two points
n_latent_samples = 7

# Define two latent points from pca space
# latent_point_1 = (24, 1) # ex1 (9, -13)
# latent_point_2 = (24, -15) # ex1 (24,-9)

# Define n latent points from latent space
crystal_types = ['NickelArsenide', 'CadmiumIodide'] #['CadmiumIodide', 'CadmiumChloride'] #['RockSalt', 'Spinel', 'ZincBlende']  ['NickelArsenide', 'CadmiumIodide', 'CadmiumChloride']
indices = [0, 4] #[14,3] #[0, 4, 0]  [0, 4, 3]
point_indices = [df_rec[(df_rec['crystalType'] == crystal_type)].index[index] for crystal_type, index in zip(crystal_types, indices)]

latent_points = [np.array(df_rec['latent_space_mean'].iloc[point_index]) for point_index in point_indices]

# Interpolation mode flag (True for PCA. False for full latent space)
pca_inter = False

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Latent samples are n_latent_samples evenly spaced points between the two points
if pca_inter:
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_pca = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_pca = np.concatenate([interp_pca, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
    
    if len(df_rec['latent_space_mean'].iloc[0]) > 2:
        interp_latent = pca.inverse_transform(interp_pca)
    else:
        interp_latent = interp_pca
    interp_latent = torch.tensor(interp_latent).float()
else:
    # Transform back to latent dimensions if latent space is more than 2D
    if (len(df_rec['latent_space_mean'].iloc[0]) > 2) and (len(latent_points[0]) == 2):
        latent_points = pca.inverse_transform(latent_points)
    
    for i in range(len(latent_points)-1):
        if i == 0:
            interp_latent = np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples) for j in range(len(latent_points[i]))]).T
        else:
            interp_latent = np.concatenate([interp_latent, np.array([np.linspace(latent_points[i][j], latent_points[i+1][j], n_latent_samples)[1:] for j in range(len(latent_points[i]))]).T], axis=0)
        
    interp_latent = torch.tensor(interp_latent).float()

n_interp_points = interp_latent.shape[0]
interp_index = torch.tensor([i for i in range(n_interp_points)])

# Create dataset
interp_dataset = TensorDataset(interp_latent, interp_index, comp_onehot.repeat(n_interp_points, 1))

# Dataloader
interp_loader = DataLoader(interp_dataset, batch_size=10, shuffle=False)

# Results dict
interp_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [25]:
# Inference
model.eval()
for batch in tqdm(interp_loader, desc='Interpolating', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    interp_point, interp_index, composition = batch
    interp_point = interp_point.to(device)
    interp_index = interp_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            interp_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    interp_results['name'].extend([f'sample_{interp_index[batch_index]}' for batch_index in range(this_batch_size)])
    interp_results['latent_point'].extend(interp_point.cpu().tolist())
    
    # Store predictions
    interp_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/interpolation_predictions/sample{interp_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {interp_index[batch_index]}.')
    

In [26]:
df_interp = pd.DataFrame(interp_results)
if len(df_interp['latent_point'].iloc[0]) == 2:
    df_interp[['ls1', 'ls2']] = df_interp['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_interp[['pc1', 'pc2']] = pca.transform(np.array(df_interp['latent_point'].values.tolist()))
df_interp['Interpolation number'] = df_interp['name'].apply(lambda x: int(x.split('_')[1]))
df_interp.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2,Interpolation number
0,sample_0,"[-0.27026253938674927, -0.4455939829349518]","[6.250069618225098, 6.2510457038879395, 10.411...","[[0.16759000718593597, 0.3382200002670288, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-0.270263,-0.445594,0
1,sample_1,"[-0.7610889077186584, -0.3081524968147278]","[6.139160633087158, 6.139456748962402, 10.0163...","[[0.16898000240325928, 0.33375000953674316, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-0.761089,-0.308152,1
2,sample_2,"[-1.2519152164459229, -0.17071101069450378]","[6.119599342346191, 6.120657444000244, 9.64372...","[[0.16989000141620636, 0.3334699869155884, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.251915,-0.170711,2
3,sample_3,"[-1.742741584777832, -0.033269524574279785]","[6.08494758605957, 6.086925983428955, 9.418951...","[[0.16787000000476837, 0.33423998951911926, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.742742,-0.033270,3
4,sample_4,"[-2.233567953109741, 0.10417196154594421]","[6.020927429199219, 6.020894527435303, 9.27038...","[[0.16776999831199646, 0.3358199894428253, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-2.233568,0.104172,4


## Plot

In [27]:
# Plot interpolation results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_samples.pdf', dpi=300)

In [28]:
# Plot the cell parameters of the interpolation with angles and lengths on two subplots (one for lengths and one for angles) that share the x-axis
df_cell_params = pd.DataFrame(df_interp['cell_parameters'].tolist(), columns=['a', 'b', 'c', 'alpha', 'beta', 'gamma'])
df_cell_params['name'] = df_interp['name']

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Lengths
sns.lineplot(data=df_cell_params[['a', 'b', 'c']], ax=ax0, zorder=10, lw=2)
ax0.set_ylabel('Length [Å]', fontsize=14, fontweight='bold')

# Angles
sns.lineplot(data=df_cell_params[['alpha', 'beta', 'gamma']], ax=ax1, zorder=10, lw=2)
ax1.set_ylabel('Angle [°]', fontsize=14, fontweight='bold')
ax1.set_xlabel('Interpolation sample #', fontsize=14, fontweight='bold')
# y-axis label and ticks on the right side
ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")

# Add vertical lines for the interpolation outside clusters and color between them
line_1 = 3.5
line_2 = 4.5
line_3 = 8.5
line_4 = 10.5
ax0.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax0.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax0.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_1, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_2, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_1, line_2, color='red', alpha=0.1, zorder=1)

ax1.axvline(x=line_3, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvline(x=line_4, color='red', linestyle='-', alpha=0.5, zorder=2)
ax1.axvspan(line_3, line_4, color='red', alpha=0.1, zorder=1)

plt.xlim(0, n_interp_points-1)

# Legends
ax0.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)
ax1.legend(loc='center left' , bbox_to_anchor=(0.01, 0.5), ncol=1)

# Make x-ticks integers
plt.xticks(np.arange(0, n_interp_points, 1))

fig.tight_layout()

# Remove space between subplots
plt.subplots_adjust(hspace=0)

plt.show()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/interpolation_cell_parameters.pdf', dpi=300)

# Sample same distribution multiple times

## Code

In [29]:
# Number of samples to generate
n_samples = 10

# Latent distribution to sample from
dist_crystal_type = 'CadmiumIodide'
crystal_type_index = 0
dist_index = df_rec[(df_rec['crystalType'] == dist_crystal_type)].index[crystal_type_index]
dist_mean = df_rec.latent_space_mean_prior.iloc[dist_index]
dist_std = df_rec.latent_space_std_prior.iloc[dist_index]

# Latent distribution from experimental data
# exp_index = 3
# dist_mean = df_exp_results.prior_mean.iloc[exp_index]
# dist_std = df_exp_results.prior_log_std.iloc[exp_index]

latent_dist = torch.distributions.Normal(loc=torch.tensor(dist_mean).float(), scale=torch.tensor(dist_std).float())

# Sample from latent distribution
latent_mean = latent_dist.mean
latent_samples = latent_dist.sample((n_samples,))
# Concatenate with latent mean
latent_samples = torch.cat((latent_mean.unsqueeze(0), latent_samples), dim=0)
sample_index = torch.tensor([i for i in range(n_samples+1)])

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

# Define dataset
sample_dataset = TensorDataset(latent_samples, sample_index, comp_onehot.repeat(n_samples+1, 1))

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
}

In [30]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition = batch
    sample_point = sample_point.to(device)
    sample_index = sample_index.to(device)
    composition = composition.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    sample_results['name'].extend([f'sample_{sample_index[batch_index]}' for batch_index in range(this_batch_size)])
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Create CIF files
    for batch_index in range(this_batch_size):
        # Prediction
        try:
            create_cif(
                cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
                cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
                cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
                filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/sample_predictions/sample{sample_index[batch_index]}',
                prediction=True,
                composition=None,
                simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
            )
        except:
            print(f'Failed to create CIF file for prediction of sample {sample_index[batch_index]}.')

In [31]:
df_sample = pd.DataFrame(sample_results)
if len(df_sample['latent_point'].iloc[0]) == 2:
    df_sample[['ls1', 'ls2']] = df_sample['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_sample[['pc1', 'pc2']] = pca.transform(np.array(df_sample['latent_point'].values.tolist()))
    
df_sample['name'][df_sample['name'] == 'sample_0'] = 'Dist. mean'
df_sample.head()

,name,latent_point,cell_parameters,cell_positions,cell_atoms,ls1,ls2
0,Dist. mean,"[-4.860902786254883, -0.4516785442829132]","[5.701595306396484, 5.699002742767334, 8.37142...","[[0.16282999515533447, 0.34073999524116516, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-4.860903,-0.451679
1,sample_1,"[-4.79807710647583, -0.27787071466445923]","[5.721805095672607, 5.718235015869141, 8.39274...","[[0.16122999787330627, 0.3424699902534485, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-4.798077,-0.277871
2,sample_2,"[-5.060468673706055, -0.050646841526031494]","[5.728972911834717, 5.718817234039307, 8.36745...","[[0.1587499976158142, 0.3494400084018707, 0.11...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-5.060469,-0.050647
3,sample_3,"[-5.15463399887085, -0.38223880529403687]","[5.696178436279297, 5.689232349395752, 8.32643...","[[0.15926000475883484, 0.34525999426841736, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-5.154634,-0.382239
4,sample_4,"[-4.4964280128479, -0.38480067253112793]","[5.722249507904053, 5.722765922546387, 8.44997...","[[0.1654299944639206, 0.3387700021266937, 0.11...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-4.496428,-0.384801


## Plot

In [32]:
# Plot sampling results in latent space
if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='ls1', y='ls2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='ls1', y='ls2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('LS dim 1')
    plt.ylabel('LS dim 2')
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=df_rec, x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    # sns.scatterplot(data=df_sample, x='pc1', y='pc2', hue='name', palette='viridis', marker='o', s=100)
    sns.scatterplot(data=df_sample[df_sample['name'] != 'Dist. mean'], x='pc1', y='pc2', color='black', marker='.', s=100, label='Samples')
    sns.scatterplot(data=df_sample[df_sample['name'] == 'Dist. mean'], x='pc1', y='pc2', color='red', marker='.', s=200, label='Dist. mean')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sampling_samples.pdf', dpi=300)

# Sample every latent distrubution and calculate loss

In [101]:
# Number of samples per point
n_samples = 100 #1000
test_data = 'validation'

# Seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Sample from each latent distribution
data_names = []
data_samples = []
data_sample_type = []
data_crystal_types = []
ground_truth_cell_parameters = []
ground_truth_cell_positions = []
ground_truth_cell_atoms = []
for data_index in range(len(df_rec)):
    latent_mean = df_rec['latent_space_mean'].iloc[data_index]
    latent_std = df_rec['latent_space_std'].iloc[data_index]
    latent_dist = torch.distributions.Normal(loc=torch.tensor(latent_mean).float(), scale=torch.tensor(latent_std).float())
    latent_samples = latent_dist.sample((n_samples,))
    
    data_samples.append(torch.tensor(latent_mean))
    data_samples.extend([latent_sample for latent_sample in latent_samples])
    
    data_names.append(f'{data_index}')
    data_names.extend([f'{data_index}_{i}' for i in range(n_samples)])
    
    data_sample_type.append('mean')
    data_sample_type.extend(['sample' for i in range(n_samples)])
    
    data_crystal_types.extend([df_rec['crystalType'].iloc[data_index] for i in range(n_samples+1)])
    
    cell_positions_padded = np.zeros((setup_json['model']['out_dim'], 3))
    cell_positions_padded[:len(df_rec['cell_positions'].iloc[data_index]), :] = np.array(df_rec['cell_positions'].iloc[data_index])
    # if setup_json['training']['simplified_atom_identities']:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 3))
    # else:
    #     cell_atoms_padded = torch.zeros((setup_json['model']['out_dim'], 119))
    cell_atoms_padded = np.zeros((setup_json['model']['out_dim'],))
    cell_atoms_padded[:len(df_rec['cell_atoms'].iloc[data_index])] = np.array(df_rec['cell_atoms'].iloc[data_index])
    
    # Ground truth
    ground_truth_cell_parameters.extend([df_rec['cell_parameters'].iloc[data_index]]*(n_samples+1))
    ground_truth_cell_positions.extend([cell_positions_padded]*(n_samples+1))
    ground_truth_cell_atoms.extend([cell_atoms_padded]*(n_samples+1))
    
data_samples = torch.stack(data_samples)
ground_truth_cell_parameters = torch.tensor(ground_truth_cell_parameters, dtype=torch.float32)
ground_truth_cell_positions = torch.tensor(ground_truth_cell_positions, dtype=torch.float32)
ground_truth_cell_atoms = torch.tensor(ground_truth_cell_atoms, dtype=torch.long)

# Define dummy composition
comp = ['Fe', 'O']
comp = Atoms(symbols=comp).get_atomic_numbers()
comp_onehot = torch.zeros(119)
comp_onehot[comp] = 1

data_index = torch.tensor(np.arange(len(data_names)))

# Define dataset
sample_dataset = TensorDataset(data_samples, data_index, comp_onehot.repeat(len(data_names), 1), ground_truth_cell_parameters, ground_truth_cell_positions, ground_truth_cell_atoms)

# Dataloader
sample_loader = DataLoader(sample_dataset, batch_size=10, shuffle=False)

# Results dict
sample_results = {
    'name': [],
    'sample_type': [],
    'crystalType': [],
    'latent_point': [],
    'cell_parameters': [],
    'cell_positions': [],
    'cell_atoms': [],
    'reconstruction_loss': [],
}
    

In [102]:
# Loss functions
loss_fn_cell_parameters = torch.nn.MSELoss()
# loss_fn_cell_positions = torch.nn.MSELoss()
# loss_fn_cell_atoms = torch.nn.CrossEntropyLoss()
# loss_fn_kld = torch.nn.KLDivLoss()
loss_fn_cell_positions = weighted_MSELoss()
loss_fn_cell_atoms = weighted_CrossEntropyLoss()
loss_fn_latent_mean = torch.nn.MSELoss()
loss_fn_latent_std = torch.nn.MSELoss()

In [103]:
# Inference
model.eval()
for batch in tqdm(sample_loader, desc='Sampling', disable=setup_json['disable_tqdm']):
    this_batch_size = len(batch[0])
    sample_point, sample_index, composition, cell_parameters_true, cell_positions_true, cell_atoms_true = batch
    sample_point = sample_point.to(device)
    sample_indeces = sample_index.to(device)
    composition = composition.to(device)
    cell_parameters_true = cell_parameters_true.to(device)
    cell_positions_true = cell_positions_true.to(device)
    cell_atoms_true = cell_atoms_true.to(device)
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms = model.decode(
            sample_point, 
            composition=composition,
        )
    
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store latent points
    for batch_index, sample_index in enumerate(sample_indeces):
        sample_results['name'].append(data_names[sample_index])
        sample_results['sample_type'].append(data_sample_type[sample_index])
        if data_crystal_types[sample_index] == 'Interpolated':
            sample_results['crystalType'].append(data_crystal_types[sample_index])#+ ' ' + str(sum(cell_atoms_true[batch_index] > 0).item()))
        else:
            sample_results['crystalType'].append(data_crystal_types[sample_index])
    
    sample_results['latent_point'].extend(sample_point.cpu().tolist())
    
    # Store predictions
    sample_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    sample_results['cell_positions'].extend(cell_positions.cpu().tolist())
    sample_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # Make loss weights
    cell_positions_weights = torch.where(cell_positions_true != -1, 1, 0).float().to(device)
    cell_atoms_weights = torch.where(cell_atoms_true != 0, 1, 0.1).float().to(device)

    # Simplify atom identities
    if setup_json['training']['simplified_atom_identities']:
        # Map atom number 0 to logit 0 (No atom)
        cell_atoms_true = torch.where(cell_atoms_true == 0, 0, cell_atoms_true)
        # Map atom numbers of ligands to logit 1 (Ligand) # [1, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53]
        for ligand in setup_json['training']['ligands']:
            cell_atoms_true = torch.where(cell_atoms_true == ligand, 1, cell_atoms_true)
        # Map all other atom numbers to logit 2 (Metal)
        cell_atoms_true = torch.where(cell_atoms_true >= 2, 2, cell_atoms_true)
    
    # Calculate reconstruction loss for each sample
    for batch_index in range(this_batch_size):
        loss_cell_parameters = loss_fn_cell_parameters(cell_parameters[batch_index], cell_parameters_true[batch_index])
        loss_cell_positions = loss_fn_cell_positions(cell_positions[batch_index], cell_positions_true[batch_index], cell_positions_weights[batch_index])
        loss_cell_atoms = loss_fn_cell_atoms(cell_atoms[batch_index], cell_atoms_true[batch_index], cell_atoms_weights[batch_index])
        
        loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
        
        # Save loss
        sample_results['reconstruction_loss'].append(loss_reconstruction.item())
        
    # # Reconstruction loss
    # loss_cell_parameters = loss_fn_cell_parameters(cell_parameters, cell_parameters_true) 
    
    # # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true) # Unweighted
    # loss_cell_positions = loss_fn_cell_positions(cell_positions, cell_positions_true, cell_positions_weights) # Weighted
    
    # # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true) # Unweighted
    # loss_cell_atoms = loss_fn_cell_atoms(cell_atoms, cell_atoms_true, cell_atoms_weights) # Weighted
    
    # loss_reconstruction = loss_cell_parameters + loss_cell_positions + loss_cell_atoms
    
    # # Save loss
    # sample_results['reconstruction_loss'].extend([loss_reconstruction.item()]*this_batch_size)


In [106]:
df_full_latent = pd.DataFrame(sample_results)
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    df_full_latent[['ls1', 'ls2']] = df_full_latent['latent_point'].apply(pd.Series)
else:
    # Perform PCA
    df_full_latent[['pc1', 'pc2']] = pca.transform(np.array(df_full_latent['latent_point'].values.tolist())
)
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Interpolated,"[-0.5473625063896179, -0.1996995508670807]","[6.269820690155029, 6.267766952514648, 10.1332...","[[0.16930000483989716, 0.33329999446868896, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.062625,-0.547363,-0.199700
1,0_0,sample,Interpolated,"[-0.4592984914779663, -0.12743398547172546]","[6.348290920257568, 6.346805095672607, 10.1719...","[[0.16946999728679657, 0.333079993724823, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.052758,-0.459298,-0.127434
2,0_1,sample,Interpolated,"[-0.5061978697776794, -0.3020046055316925]","[6.235864639282227, 6.236192226409912, 10.1707...","[[0.16582000255584717, 0.3311600089073181, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.082407,-0.506198,-0.302005
3,0_2,sample,Interpolated,"[-0.5163573622703552, -0.25968480110168457]","[6.252856731414795, 6.250967979431152, 10.1611...","[[0.16832000017166138, 0.33142000436782837, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.068893,-0.516357,-0.259685
4,0_3,sample,Interpolated,"[-0.5493307709693909, -0.27766862511634827]","[6.229388236999512, 6.22728967666626, 10.14525...","[[0.1682800054550171, 0.3315599858760834, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.064558,-0.549331,-0.277669


In [108]:
# Plot sampling results in latent space
if 'Interpolation' in model_path:
    custom_palette = ['k', sns.color_palette('tab20')[4], sns.color_palette('tab20')[7]]
    custom_markers = ['o', (4,0,0), (4,1,45)]
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette=custom_palette, binwidth=0.05, alpha=0.75)
        
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette, markers=custom_markers)
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & (df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette=custom_palette[1:], markers=custom_markers[1:])
        # sns.scatterplot(data=df_full_latent[(df_full_latent['sample_type'] == 'mean') & ~(df_full_latent['crystalType'].isin(['NickelArsenide', 'CadmiumIodide']))], x='ls1', y='ls2', hue='crystalType', marker='o', s=50, palette='viridis')
        
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
else:
    if len(df_rec['latent_space_mean'].iloc[0]) == 2:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)    
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', s=50, palette='tab20')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
        plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    else:
        plt.figure(figsize=(10, 8))
        sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
        # sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', palette='tab20', s=50)
        sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
        plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        plt.xlabel('PC 1', fontsize=14, fontweight='bold')
        plt.ylabel('PC 2', fontsize=14, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        # plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_density.svg', dpi=300, transparent=True)

In [59]:
# Plot interpolation results over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='ls1', y='ls2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp, x='pc1', y='pc2', hue='Interpolation number', marker='o', s=50, palette='icefire', edgecolor='black')
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.pdf', dpi=300)

# Save svg for reconstruction figure

# Remove axis border, labels and ticks
plt.axis('off')
# Remove legend
plt.legend().remove()
# Save
plt.tight_layout()
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_sample_density.svg', dpi=300, transparent=True)

In [60]:
df_exp_results_2 = df_exp_results.copy()
df_exp_results_2['Label'] = ''
for i in range(len(df_exp_results_2)):
    if df_exp_results_2['composition'].iloc[i] == 'IrO2':
        df_exp_results_2['Label'].iloc[i] = 'IrO2 (Rutile)'
    elif df_exp_results_2['composition'].iloc[i] == 'CeO2':
        df_exp_results_2['Label'].iloc[i] = 'CeO2 (Fluorite)'
    else:
        df_exp_results_2['Label'].iloc[i] = 'HEA (Spinel)'
        
# Remove index 1
df_exp_results_2.drop(index=1, inplace=True)

In [61]:
# Plot experimental data over the sample density

if len(df_rec['latent_space_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='ls1', y='ls2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='ls1', y='ls2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_exp_results_2, x='pc1', y='pc2', color='black', style='Label', s=100)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_experimental_sample_density.pdf', dpi=300)

In [62]:
df_exp_results

,composition,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_positions,cell_atoms,ls1,ls2,ground_truth_crystal_type
0,IrO2,"[[0.0], [0.0007761651650071144], [0.0015256106...","[-1.593813180923462, -0.6405230164527893]","[0.09613613039255142, 0.09387928247451782]","[-1.717536449432373, -0.6064298152923584]","[5.897527694702148, 5.899387836456299, 9.38331...","[[0.16680000722408295, 0.3352000117301941, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.593813,-0.640523,Rutile
1,IrO2,"[[0.0], [0.0005720379413105547], [0.0011345194...","[-2.1372663974761963, -0.21823544800281525]","[0.11543595790863037, 0.11163424700498581]","[-2.1938061714172363, -0.07631607353687286]","[5.989924907684326, 5.992015838623047, 9.24749...","[[0.167480006814003, 0.33520999550819397, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-2.137266,-0.218235,Rutile
2,IrO2,"[[0.0], [-0.000292735145194456], [-0.000565890...","[-1.45340895652771, -0.6876246333122253]","[0.09248798340559006, 0.09030269831418991]","[-1.524922490119934, -0.6281325221061707]","[5.898794174194336, 5.898250579833984, 9.50557...","[[0.16633999347686768, 0.334089994430542, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.453409,-0.687625,Rutile
3,CeO2,"[[0.0], [-0.000268803967628628], [-0.000530091...","[0.19988952577114105, 0.21497651934623718]","[0.06774086505174637, 0.07994374632835388]","[0.14612111449241638, 0.13758671283721924]","[6.847782135009766, 6.845365047454834, 10.4790...","[[0.1694899946451187, 0.33445000648498535, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.199890,0.214977,Fluorite
4,CeO2,"[[0.0], [-0.0003009798820130527], [-0.00059319...","[-0.004733748733997345, -0.8593271374702454]","[0.06259611994028091, 0.07178624719381332]","[0.03918557986617088, -0.8405420780181885]","[6.002366542816162, 6.010383129119873, 9.73057...","[[0.08987999707460403, 0.17062999308109283, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-0.004734,-0.859327,Fluorite
5,CeO2,"[[0.0], [0.0007327766506932676], [0.0014382996...","[0.23670463263988495, 0.8518111109733582]","[0.0790347158908844, 0.09334874153137207]","[0.24053041636943817, 0.8878253102302551]","[7.091526508331299, 7.09077262878418, 10.63883...","[[0.1697400063276291, 0.33292001485824585, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.236705,0.851811,Fluorite
6,Cu0.5Ni0.5Cr0.66Mn0.66Fe0.66O4,"[[0.0], [-1.6920250345719978e-05], [-3.3054449...","[-1.3475160598754883, -2.117427349090576]","[0.16324155032634735, 0.15959382057189941]","[-1.202986717224121, -2.0386013984680176]","[5.059391498565674, 5.087088108062744, 9.91576...","[[0.0714699998497963, 0.15647999942302704, 0.0...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",-1.347516,-2.117427,Spinel
7,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [0.001129354815930128], [0.00220868061...","[-1.6955444812774658, -1.5706487894058228]","[0.19478626549243927, 0.17367976903915405]","[-1.672194004058838, -1.4464322328567505]","[5.570705890655518, 5.5693769454956055, 9.4792...","[[0.16436000168323517, 0.3342599868774414, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.695544,-1.570649,Spinel
8,Co0.33Zn0.33Ni0.33CrMnO4,"[[0.0], [5.6757980928523466e-05], [0.000110909...","[-1.6866625547409058, -1.597379207611084]","[0.19687506556510925, 0.17541103065013885]","[-1.623341679573059, -1.5798455476760864]","[5.546947479248047, 5.549094200134277, 9.47320...","[[0.16401000320911407, 0.3332200050354004, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1.686663,-1.597379,Spinel
9,Co0.33Cu0.33Ni0.33CrMnO4,"[[0.0], [-1.0808051229105331e-05], [-2.1100040...","[-1.6533193588256836, -2.0307412147521973]","[0.16366592049598694, 0.15191011130809784]","[-1.6093007326126099, -1.9724514484405518]","[5.399789333343506, 5.434008598327637, 9.44117...","[[0.14959999918937683, 0.2658199965953827, 0.0...","[1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, ...",-1.653319,-2.030741,Spinel


In [63]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Interpolated,"[-0.5473625063896179, -0.1996995508670807]","[6.269820690155029, 6.267766952514648, 10.1332...","[[0.16930000483989716, 0.33329999446868896, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.062625,-0.547363,-0.199700
1,0_0,sample,Interpolated,"[-0.4592984914779663, -0.12743398547172546]","[6.348290920257568, 6.346805095672607, 10.1719...","[[0.16946999728679657, 0.333079993724823, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.052758,-0.459298,-0.127434
2,0_1,sample,Interpolated,"[-0.5061978697776794, -0.3020046055316925]","[6.235864639282227, 6.236192226409912, 10.1707...","[[0.16582000255584717, 0.3311600089073181, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.082407,-0.506198,-0.302005
3,0_2,sample,Interpolated,"[-0.5163573622703552, -0.25968480110168457]","[6.252856731414795, 6.250967979431152, 10.1611...","[[0.16832000017166138, 0.33142000436782837, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.068893,-0.516357,-0.259685
4,0_3,sample,Interpolated,"[-0.5493307709693909, -0.27766862511634827]","[6.229388236999512, 6.22728967666626, 10.14525...","[[0.1682800054550171, 0.3315599858760834, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.064558,-0.549331,-0.277669


In [64]:
# Plot latent space with surface showing the loss
from scipy.interpolate import griddata
import matplotlib.colors as colors

# Decide on contour column
contour_column = 'reconstruction_loss'

# Get min and max values
min_loss = df_full_latent[contour_column].min()
max_loss = df_full_latent[contour_column].max()

# Plot
if len(df_full_latent['latent_point'].iloc[0]) == 2:
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['ls1'].min()-0.1, df_full_latent['ls1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['ls2'].min()-0.1, df_full_latent['ls2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['ls1'], df_full_latent['ls2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='ls1', y='ls2', hue='crystalType', style='crystalType', palette='tab20', s=50)
    plt.colorbar()
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
else:
    
    # Interpolate z values to a grid from points
    xi = np.linspace(df_full_latent['pc1'].min()-0.1, df_full_latent['pc1'].max()+0.1, 1000)
    yi = np.linspace(df_full_latent['pc2'].min()-0.1, df_full_latent['pc2'].max()+0.1, 1000)
    zi = griddata((df_full_latent['pc1'], df_full_latent['pc2']), df_full_latent[contour_column], (xi[None,:], yi[:,None]), method='linear', fill_value=np.nan)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xi, yi, zi, 200, cmap='viridis')#, norm=colors.LogNorm())
    sns.scatterplot(data=df_full_latent[df_full_latent['sample_type'] == 'mean'], x='pc1', y='pc2', hue='crystalType', style='crystalType', s=50, palette='tab20')
    plt.colorbar()
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()
    
plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_sample_loss.pdf', dpi=300)	

In [65]:
df_full_latent.head()

,name,sample_type,crystalType,latent_point,cell_parameters,cell_positions,cell_atoms,reconstruction_loss,ls1,ls2
0,0,mean,Interpolated,"[-0.5473625063896179, -0.1996995508670807]","[6.269820690155029, 6.267766952514648, 10.1332...","[[0.16930000483989716, 0.33329999446868896, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.062625,-0.547363,-0.199700
1,0_0,sample,Interpolated,"[-0.4592984914779663, -0.12743398547172546]","[6.348290920257568, 6.346805095672607, 10.1719...","[[0.16946999728679657, 0.333079993724823, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.052758,-0.459298,-0.127434
2,0_1,sample,Interpolated,"[-0.5061978697776794, -0.3020046055316925]","[6.235864639282227, 6.236192226409912, 10.1707...","[[0.16582000255584717, 0.3311600089073181, 0.1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.082407,-0.506198,-0.302005
3,0_2,sample,Interpolated,"[-0.5163573622703552, -0.25968480110168457]","[6.252856731414795, 6.250967979431152, 10.1611...","[[0.16832000017166138, 0.33142000436782837, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.068893,-0.516357,-0.259685
4,0_3,sample,Interpolated,"[-0.5493307709693909, -0.27766862511634827]","[6.229388236999512, 6.22728967666626, 10.14525...","[[0.1682800054550171, 0.3315599858760834, 0.12...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0.064558,-0.549331,-0.277669


In [66]:
# Plot all samples in 3d if latent space is 3D

# if len(df_rec['latent_space_mean'].iloc[0]) == 3:
    
#     df_crystal = df_full_latent.copy()
#     df_crystal[['ls1', 'ls2', 'ls3']] = df_crystal['latent_point'].apply(pd.Series)
    
#     fig = px.scatter_3d(df_crystal, x='ls1', y='ls2', z='ls3', color='crystalType', symbol='crystalType', hover_data=['name', 'ls1', 'ls2', 'ls3', 'reconstruction_loss', 'cell_parameters'], color_discrete_sequence=px.colors.qualitative.Dark24)
    
#     fig.update_layout(
#         scene = dict(
#             xaxis_title='LS dim 1',
#             yaxis_title='LS dim 2',
#             zaxis_title='LS dim 3',
#         ),
#         margin=dict(l=0, r=0, t=0, b=0),
#     )
    # fig.show()

# Interpolation dataset input to model

In [ ]:
# Load CHILI dataset
dataset_interp = CHILI(
    root=setup_json['data']['root'],
    dataset='CHILI-Interpolation_v2',
    graph_type=setup_json['data']['graph_type'],
)

# Load/create data splits
try:
    # Load existing data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
except FileNotFoundError:
    # Create new data split
    dataset_interp.create_data_split(
        test_size=setup_json['data']['split']['test'],
        validation_size=setup_json['data']['split']['validation'],
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
    # Load data split
    dataset_interp.load_data_split(
        split_strategy=setup_json['data']['split_strategy'],
        stratify_on=setup_json['data']['stratify_column'],
    )
    
# Dataloader
# data_loader_interp = DataLoader(dataset_interp.validation_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
# data_loader_interp = DataLoader(dataset_interp.train_set, batch_size=setup_json['data']['batch_size'], shuffle=False)
data_loader_interp = DataLoader(dataset_interp.test_set, batch_size=setup_json['data']['batch_size'], shuffle=False)

In [ ]:
interp_data_results = {
    'name': [],
    'crystal_type': [],
    'pdf': [],
    'prior_mean': [],
    'prior_log_std': [],
    'z_sample': [],
    'cell_parameters': [],
    'cell_atoms': [],
    'cell_positions': [],
}
    

In [ ]:
# Inference
model.eval()
for batch in tqdm(data_loader_interp, desc='Inference', disable=setup_json['disable_tqdm']):
    batch = batch.to(device)
    this_batch_size = batch.batch.amax().item() + 1
    
    # Normalize scattering
    pdf = batch.y['xPDF'][:,1,:].unsqueeze(-1)
    if setup_json['data']['normalize_scattering']:
        # Normalize so highest peak in each sample is 1
        pdf /= torch.amax(pdf, dim=1, keepdim=True)[0]
    
    # Composition conditioning
    composition = torch.zeros(this_batch_size, 119).to(device)
    elements_in_batch = batch.y['atomic_species'].long()
    index_counter = 0
    for i in range(this_batch_size):
        n_elements = batch.y['n_atomic_species'][i]
        composition[i, elements_in_batch[index_counter:index_counter + n_elements]] = 1
        index_counter += n_elements
    composition[:, 0] = 1 
    
    with torch.no_grad():
        cell_parameters, cell_positions, cell_atoms, prior_mean, prior_log_std, z_sample = model.predict(
            pdf, 
            composition,
        )
        
    # Denormalize cell parameters
    if setup_json['data']['normalize_cell_parameters']:
        cell_parameters = (cell_parameters * cell_stds) + cell_means
    
    # Rounding positions to 5 decimals
    cell_positions = torch.round(cell_positions, decimals=5)
    
    # Store names
    interp_data_results['name'].extend(batch.data_id)
    interp_data_results['crystal_type'].extend(batch.y['crystal_type'])
    
    # Store PDF
    interp_data_results['pdf'].extend(pdf.cpu().tolist())
    
    # Store latent representation
    interp_data_results['prior_mean'].extend(prior_mean.cpu().tolist())
    interp_data_results['prior_log_std'].extend(prior_log_std.cpu().tolist())
    interp_data_results['z_sample'].extend(z_sample.cpu().tolist())
    
    # Store predictions
    interp_data_results['cell_parameters'].extend(cell_parameters.cpu().tolist())
    interp_data_results['cell_positions'].extend(cell_positions.cpu().tolist())
    interp_data_results['cell_atoms'].extend(torch.argmax(cell_atoms, dim=2).cpu().tolist())
    
    # # Create CIF files
    # for batch_index in range(this_batch_size):
    #     # Prediction
    #     try:
    #         create_cif(
    #             cell_params = cell_parameters[batch_index].detach().cpu().numpy(),
    #             cell_positions = cell_positions[batch_index].detach().cpu().numpy(),
    #             cell_atoms = cell_atoms[batch_index].detach().cpu().numpy(),
    #             filename = f'{setup_json["model_root"]}{setup_json["experiment_name"]}/experimental_predictions/{data_filepath[data_filepath_index[batch_index]].split("/")[-1].split(".")[0]}',
    #             prediction=True,
    #             composition=data_composition_string[composition_string_index[batch_index]],
    #             simplified_atom_identities=setup_json['training']['simplified_atom_identities'],
    #         )
    #     except:
    #         print(f'Failed to create CIF file for prediction of {data_composition_string[composition_string_index[batch_index]]}.')
    
df_interp_data_results = pd.DataFrame(interp_data_results)
if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    df_interp_data_results[['ls1', 'ls2']] = df_interp_data_results['prior_mean'].apply(pd.Series)
else:
    # Perform PCA
    df_interp_data_results[['pc1', 'pc2']] = pca.transform(np.array(df_interp_data_results['prior_mean'].values.tolist()))
    
df_interp_data_results['Step'] = 0
df_interp_data_results['Step'][df_interp_data_results['crystal_type'] == 'interpolated'] = df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated']['name'].str.split('_').str[-3].str[-1].astype(int)

df_interp_data_results.head()

,name,crystal_type,pdf,prior_mean,prior_log_std,z_sample,cell_parameters,cell_atoms,cell_positions,ls1,ls2,Step
0,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.000650433823466301], [-0.001385843...","[0.5236246585845947, -1.6123684644699097]","[0.056431304663419724, 0.11883806437253952]","[0.6096026301383972, -1.7731014490127563]","[4.470189094543457, 4.3849687576293945, 2.7454...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[[-0.03392000123858452, 0.0002800000074785203,...",0.523625,-1.612368,3
1,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.00041956984205171466], [-0.0008662...","[0.3216722309589386, 1.8383911848068237]","[0.11844587326049805, 0.1398300677537918]","[0.4761958420276642, 1.8483673334121704]","[3.464792251586914, 3.4860076904296875, 5.3488...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 2, 1, ...","[[0.010060000233352184, -0.019020000472664833,...",0.321672,1.838391,3
2,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [0.00019362939929123968], [0.000212970...","[0.4139220714569092, -1.583296775817871]","[0.060027170926332474, 0.09942282736301422]","[0.3750808537006378, -1.6341190338134766]","[4.552481174468994, 4.547699928283691, 2.85897...","[2, 1, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 2, 2, ...","[[0.005770000163465738, -0.02111000008881092, ...",0.413922,-1.583297,3
3,NickelArsenide_V,NickelArsenide,"[[0.0], [-0.0011531105265021324], [-0.00234128...","[0.8094152212142944, 0.6322392225265503]","[0.0653667077422142, 0.07735161483287811]","[0.7035900950431824, 0.5739375352859497]","[3.061583995819092, 3.0408270359039307, 5.2172...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.002850000048056245, -0.00803999975323677, ...",0.809415,0.632239,0
4,interpolated_NickelArsenide_to_CadmiumIodide_s...,interpolated,"[[0.0], [-0.0004505524702835828], [-0.00092623...","[0.8206459879875183, 0.24802753329277039]","[0.06555262207984924, 0.07961246371269226]","[0.8808557987213135, 0.3152904510498047]","[3.978081226348877, 3.965078592300415, 5.70301...","[2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.017069999128580093, 0.012430000118911266, ...",0.820646,0.248028,3


## Plot

In [ ]:
# Plot interpolation points in latent space

if len(df_interp_data_results['prior_mean'].iloc[0]) == 2:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='ls1', y='ls2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp_data_results, x='ls1', y='ls2', color='black', marker='o', s=50)
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=50, label='NickelArsenide')
    # sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='ls1', y='ls2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=50, label='CadmiumIodide')
    # sns.histplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', palette='viridis', binwidth=0.02, alpha=1, kde=True)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='ls1', y='ls2', hue='Step', marker='.', palette='viridis', s=150, edgecolor=None, alpha=0.75)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('Latent space dim 1', fontsize=14, fontweight='bold')
    plt.ylabel('Latent space dim 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
else:
    plt.figure(figsize=(10, 8))
    sns.histplot(data=df_full_latent[df_full_latent['sample_type'] != 'mean'], x='pc1', y='pc2', hue='crystalType', palette='tab20', binwidth=0.05, alpha=0.75)
    # sns.scatterplot(data=df_interp_data_results, x='pc1', y='pc2', color='black', marker='o', s=50)
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'NickelArsenide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[4], marker=(4,0,0), s=75, label='NickelArsenide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'CadmiumIodide'], x='pc1', y='pc2', color=sns.color_palette('tab20')[7], marker=(4,1,45), s=75, label='CadmiumIodide')
    sns.scatterplot(data=df_interp_data_results[df_interp_data_results['crystal_type'] == 'interpolated'], x='pc1', y='pc2', hue='Step', marker='.', palette='viridis', s=75, edgecolor=None)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xlabel('PC 1', fontsize=14, fontweight='bold')
    plt.ylabel('PC 2', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    # plt.show()
# plt.legend(title='Interpolation #', loc='center left', bbox_to_anchor=(1, 0.5), ncol=1)
plt.tight_layout()

plt.savefig(f'{setup_json["model_root"]}{setup_json["experiment_name"]}/paper_figures/latent_space_interpolation_dataset.pdf', dpi=300)
